# 📗 관측성과 디버깅

앞 시간에 만든 성찰 루프는 모델을 **여러 번** 호출합니다. 잘 도는 것 같아도 실제 서비스로 올리면 질문이 생깁니다. **토큰(요금)을 얼마나 썼지? 어디서 느려지지? 답이 왜 저렇게 나왔지?** 이걸 눈으로 볼 수 없으면 고칠 곳도 찾지 못합니다.

이번 시간엔 에이전트 안에서 무슨 일이 있었는지 알아내는 **관측성(observability)** 을 **Langfuse** 로 다룹니다. 실행을 대시보드에 남기고, **토큰과 비용**을 직접 계산해 서버 값과 맞춰 보고, 프롬프트를 **버전**으로 관리하고, 여러 호출을 **세션**으로 묶고, 마지막으로 기록을 읽어 **틀어진 답을 고칩니다**. 관측할 값(토큰·지연)은 **실제 호출에서만 나오므로** 오늘도 본인 `OPENAI_API_KEY` 로 진행합니다.

> **오늘은 Langfuse 키도 필요합니다.** 관측성은 대시보드에 실제로 기록이 쌓이는 것을 **눈으로 봐야** 배울 수 있어, 전송을 끄면 이 단원이 성립하지 않습니다. [cloud.langfuse.com](https://cloud.langfuse.com) 무료 가입(신용카드 불필요) → 프로젝트 생성 → Settings → API Keys 에서 public·secret 키를 복사해 `.env` 의 `LANGFUSE_PUBLIC_KEY`·`LANGFUSE_SECRET_KEY`·`LANGFUSE_BASE_URL` 에 채우세요. 무료 한도(월 5만 건)로 이 수업은 충분합니다.

## ⏪ 복습

- **성찰 루프**: `generate → critique → revise` 를 임계 점수·최대 반복까지 돌려 품질을 올렸습니다.
- **메시지 기록**(에이전트 단원): 메시지 목록(Human→AI→Tool→AI)으로 **무엇이 언제 불렸는지** 읽었습니다.
- **도구(`@tool`)와 docstring**(에이전트 단원): 모델은 **설명을 읽고** 어떤 도구를 부를지 정했습니다.

**오늘의 목표**

- [ ] **관측성이 무엇이고 무엇을 기록하는지** 설명하고, 관측 도구(**Langfuse**·**LangSmith**)를 고르는 기준과 Langfuse 의 **trace·span·generation** 구조를 안다.
- [ ] **CallbackHandler** 를 호출에 붙여 실행을 **대시보드에서 확인**한다.
- [ ] **usage_metadata** 로 토큰을 읽고 **공식 요금표로 비용을 계산해**, Langfuse 가 계산한 값과 **맞춰 본다**.
- [ ] **프롬프트를 Langfuse 에 버전으로 올리고**(`create_prompt`) **라벨·버전으로 불러와**(`get_prompt`) 실행에 쓰고, 그 버전을 **trace 에 연결**한다.
- [ ] 여러 호출을 **한 세션으로 묶어**(`langfuse_session_id`) 대화 단위로 들여다본다.
- [ ] **메시지 기록으로 라우팅 실패를 재현하고**, 도구 설명을 고쳐 **바뀌는 것을 확인**한다.

In [ ]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우
load_dotenv("../../.env") # 교안 폴더 안의 정답 폴더에서 실행하는 경우
load_dotenv("../../../.env") # 교안 폴더 안의 과제/정답 에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 - 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 오늘 쓸 모델 - 실행만 하세요(LangChain 기본 단원에서 만든 것과 같습니다).
from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

print('모델 준비 완료:', type(model).__name__)

---
# 1. 관측성(observability)의 개념과 목적

## 관측성이란 무엇인가
**관측성**은 **밖에 남긴 기록만 보고 안에서 무슨 일이 있었는지 알아낼 수 있는 성질**입니다. 핵심은 **기록을 미리 남겨 둔다**는 점입니다. 실행이 끝난 뒤에 "그때 토큰을 얼마나 썼지?" 하고 물어도 남겨 둔 것이 없으면 되찾을 방법이 없습니다. 코드를 다시 열어 `print` 를 심고 재현하지 않아도 답이 나오면, 그 시스템은 관측 가능한 것입니다.

| 기록 | 무엇인가 | 이 단원에서 보는 것 |
|---|---|---|
| **로그(log)** | 무슨 일이 있었는지 시간순으로 적어 둔 글줄 | 보낸 프롬프트와 받은 응답 |
| **지표(metric)** | 세어서 모아 둔 수 | 토큰 수, 지연 시간 |
| **트레이스(trace)** | 요청 하나가 거쳐 간 단계를 순서대로 묶은 기록 | 어떤 도구가 어떤 인자로 불렸는지 |

<img src="../images/observability_concept.png" width="880">

## 모니터링과 무엇이 다른가
**모니터링(monitoring)** 은 미리 정해 둔 지표를 지켜보는 것입니다("실패율이 1%를 넘으면 알린다"). 관측성은 **미리 예상하지 못한 질문에 사후로 답할 수 있는 상태**를 만드는 것입니다("어제 그 요청은 왜 3만 토큰을 썼나"). 모니터링이 **무언가 잘못됐다**를 알려 주고, 관측성이 **무엇이 왜 잘못됐나**에 답합니다.

## 관측 도구 고르기: Langfuse 와 LangSmith
LLM 애플리케이션 관측 도구로 가장 많이 쓰이는 둘입니다. 하는 일은 같습니다. 관측을 켜 두면 프롬프트·응답·토큰·지연·도구 호출·성공 여부가 자동으로 서버에 쌓이고, 웹 대시보드에서 요청 하나를 열어 단계별로 들여다봅니다.

| 항목 | **Langfuse** (이 단원에서 사용) | **LangSmith** |
|---|---|---|
| 만든 곳 | 독립 오픈소스 프로젝트 | LangChain 팀 |
| 켜는 법 | 콜백을 `config` 에 넘김(2절) | 환경변수 `LANGSMITH_TRACING=true` 와 API 키 |
| LangChain 밖의 코드 | 데코레이터 `@observe` | 데코레이터 `@traceable` |
| 사내 서버에 직접 설치 | 무료(오픈소스) | Enterprise 플랜에서만 |
| 무료 한도 | 월 5만 건 | 1인, 월 5천 트레이스, 14일 보관 |

**이 수업이 Langfuse 를 쓰는 이유**

- **직접 설치할 수 있습니다.** 기록에는 프롬프트와 응답이 글자 그대로 남고, 거기엔 고객 데이터가 섞입니다. 밖으로 내보낼 수 없는 회사에서는 사내 서버에 올릴 수 있는지가 곧 도입 가능 여부입니다.
- **특정 프레임워크에 묶이지 않습니다.** 나중에 LangChain 을 걷어내고 OpenAI SDK 만 써도 같은 대시보드를 그대로 씁니다.
- **프롬프트 버전 관리가 같은 도구 안에 있습니다.** 4절에서 프롬프트를 올리고 불러오는 일까지 한 화면에서 합니다.
- **무료 한도가 넉넉합니다.** 카드 없이 가입해 이 단원 실습을 전부 돌릴 수 있습니다.

LangSmith 가 떨어진다는 뜻은 아닙니다. LangChain·LangGraph 만 쓰고 클라우드에 기록을 두어도 되는 팀이라면 환경변수만 넣으면 켜지는 LangSmith 가 더 빠릅니다. 도구는 갈아 끼울 수 있고, 이 단원에서 배울 것은 도구 사용법보다 **기록을 어떤 계층으로 남기고 무엇을 봐야 하는가**입니다.

---
# 2. Langfuse 구조와 연동

## 세 가지 단위
Langfuse 는 실행을 **계층**으로 기록합니다.

| 단위 | 뜻 | 예 |
|---|---|---|
| **trace** | 요청 하나의 전체 실행 | "리포트 자동 생성 1건" |
| **span** | trace 안의 한 단계 | "비평 단계" |
| **generation** | 모델 호출 하나 | "critique 프롬프트 → 응답(토큰 42)" |

즉 하나의 **trace** 안에 여러 **span** 이 있고, span 안에 실제 모델 호출인 **generation** 이 담깁니다. 이 계층 덕분에 "이 요청은 총 몇 토큰, 어느 단계가 느렸나"를 한눈에 봅니다.

<img src="../images/trace_span_generation.png" width="880">

## 연동: CallbackHandler
LangChain 호출에 **콜백**을 붙이면 실행이 자동으로 Langfuse 로 전송됩니다. 우리가 고칠 것은 **호출 코드가 아니라 `config` 한 자리**입니다. 관측을 붙이려고 로직을 바꾸지 않아도 된다는 것이 콜백 방식의 핵심입니다.

In [ ]:
# [제공 코드] Langfuse 콜백 준비 - 실행만 하세요.
# 핸들러는 .env 의 LANGFUSE_* 를 스스로 읽습니다 - 키를 인자로 넘기지 않습니다.
import os

from langfuse import get_client
from langfuse.langchain import CallbackHandler

# 키를 먼저 확인합니다 - 핸들러를 만든 뒤에 검사하면 이 안내가 묻힙니다.
if not (os.getenv('LANGFUSE_PUBLIC_KEY') and os.getenv('LANGFUSE_SECRET_KEY')):
    raise RuntimeError('Langfuse 키를 찾지 못했습니다. 일차 폴더 .env 의 LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY 를 채우고 커널을 재시작하세요.')

handlers = [CallbackHandler()]

# 키가 '있지만 틀린' 경우 langfuse 는 전송만 조용히 실패합니다 - 그래서 인증을 여기서 확인합니다.
try:
    authenticated = get_client().auth_check()
except Exception:
    authenticated = False
if not authenticated:
    raise RuntimeError('Langfuse 인증에 실패했습니다. 일차 폴더 .env 의 키와 LANGFUSE_BASE_URL 을 확인하고 커널을 재시작하세요.')

print('Langfuse 관측 켜짐 - 이제부터의 호출이 대시보드로 전송됩니다')

이제 아무 호출에나 `config={'callbacks': handlers}` 를 붙이면 됩니다.

**확인 기준**: 아래 셀을 실행한 뒤 [cloud.langfuse.com](https://cloud.langfuse.com) 의 **Tracing → Traces** 를 열어 방금 그 호출이 **trace 한 건**으로 올라왔는지 보세요. 그 안에 프롬프트·응답·토큰 수가 그대로 들어 있습니다. (전송은 잠깐의 지연이 있을 수 있으니 안 보이면 새로고침하세요.)

In [ ]:
# 호출 코드는 앞 단원과 똑같다 - config 한 줄이 늘었을 뿐인데 이 실행이 대시보드에 남는다
response = model.invoke('한 문장으로: 관측성이 중요한 이유는?',
                        config={'callbacks': handlers})
print(response.text)

### 🖐️ 함께 따라하기: 콜백을 붙여 호출하기

`'좋은 로그의 조건 한 가지'` 를 묻는 호출에 `config={'callbacks': handlers}` 를 붙여 답을 출력해 보세요.

**확인 기준**: 답이 출력되고, 대시보드의 **Tracing → Traces** 에 trace 가 **한 건 더** 쌓입니다. 호출 코드에서 달라진 것은 `config` 한 자리뿐입니다. **관측은 코드를 고쳐 넣는 것이 아니라 얹는 것**이라는 게 이 연습의 요점입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) '좋은 로그의 조건 한 가지' 를 model.invoke 로 묻는다
# 2) 그때 config 에 callbacks=handlers 를 넣는다
# 3) 응답의 .text 를 출력한다

### ✅ 바로 확인 퀴즈

**1.** Langfuse 의 **trace / span / generation** 을 크기 순(큰 것 → 작은 것)으로 나열하면?

<details><summary>정답 보기</summary>

**trace(요청 전체) → span(단계) → generation(모델 호출 하나)** 순입니다.

</details>

**2.** 관측을 붙일 때 호출 로직이 아니라 `config` 에 콜백을 넘기는 방식이면 무엇이 좋은가요?

<details><summary>정답 보기</summary>

**기존 코드를 고치지 않고** 관측을 켜고 끌 수 있습니다. 관측을 붙이려고 체인·에이전트 내부를 손대면 관측 때문에 로직이 바뀌는 사고가 생깁니다.

</details>

---
# 3. 토큰과 비용: 내 계산과 Langfuse 값 맞춰 보기

관측의 기본은 **얼마나 썼나**입니다. 이 절은 세 걸음입니다. 응답에 실려 온 **토큰을 읽고**, 공식 요금표로 **비용을 계산하고**, 같은 호출을 **Langfuse 가 계산해 둔 값과 맞춰** 봅니다.

## 3.1 토큰은 응답에 실려 온다
따로 재는 것이 아니라 **읽는** 것입니다. 모델 응답의 `usage_metadata` 에 입력·출력·전체 토큰이 들어 있습니다.

In [ ]:
# 토큰 수는 응답에 딸려 온다 - 따로 재는 게 아니라 '읽는' 것이다
response = model.invoke('한국의 수도는?')
print('usage_metadata:', response.usage_metadata)
print('전체 토큰:', response.usage_metadata['total_tokens'])

In [ ]:
# 여러 호출의 토큰을 더해 총 사용량을 집계한다 - 루프를 한 번 더 돌리면 요금도 그만큼 늘어난다
questions = ['1+1은?', '지구에서 가장 큰 바다는?', '무지개는 몇 색?']
total_tokens = 0
for question in questions:
    answer = model.invoke(question)
    total_tokens += answer.usage_metadata['total_tokens']
print('세 번 호출 총 토큰:', total_tokens)

## 3.2 공식 요금표로 비용을 계산한다

토큰은 **100만 개 단위**로 값이 매겨지고, **입력과 출력의 단가가 다릅니다**(출력이 더 비쌉니다). 그래서 비용은 이렇게 나옵니다.

```
비용 = 입력토큰 ÷ 1,000,000 × 입력단가 + 출력토큰 ÷ 1,000,000 × 출력단가
```

우리가 쓰는 `gpt-4o-mini` 의 단가는 OpenAI 요금 문서에 적힌 값입니다.

| 항목 | 100만 토큰당 |
|---|---|
| 입력 | $0.15 |
| 출력 | $0.60 |

> 단가는 **모델마다 다르고 바뀝니다.** 코드에 숫자를 박아 두었다면 [OpenAI 요금 문서](https://platform.openai.com/docs/pricing)에서 확인하고 고쳐야 합니다.

In [ ]:
# 100만 토큰당 단가(달러) - 출처: OpenAI 요금 문서의 gpt-4o-mini 항목
PRICE_PER_1M = {'input': 0.15, 'output': 0.60}


def cost_usd(usage):
    """usage_metadata 를 받아 그 호출의 비용을 달러로 돌려준다."""
    # 입력과 출력은 단가가 다르므로 따로 곱해서 더한다
    input_cost = usage['input_tokens'] / 1000000 * PRICE_PER_1M['input']
    output_cost = usage['output_tokens'] / 1000000 * PRICE_PER_1M['output']
    return input_cost + output_cost


# 아주 작은 값이라 지수 표기(e-06)로 찍힌다 - 자리수를 정해 읽기 좋게 만든다
print('방금 그 호출 비용: $', f'{cost_usd(response.usage_metadata):.8f}')

## 3.3 Langfuse 가 계산해 둔 값과 맞춰 보기

Langfuse 도 실행이 올라오면 **같은 방식으로 비용을 계산해 둡니다.** 두 값이 맞으면 우리 계산이 맞다는 뜻이고, 어긋나면 **단가표가 다르다**는 뜻입니다(모델을 다른 것으로 부르고 있었거나, 요금이 바뀐 경우).

서버에서 값을 꺼내려면 그 실행의 **trace id** 가 필요합니다. 호출을 span 하나로 감싸면 그 안에서 id 를 얻을 수 있습니다.

In [ ]:
# 호출을 span 으로 감싸면 그 실행의 trace id 를 그 자리에서 얻을 수 있다
from langfuse import get_client

langfuse = get_client()

with langfuse.start_as_current_observation(as_type='span', name='비용 확인'):
    checked = model.invoke('한 문장으로: 관측성이 왜 필요한가?',
                           config={'callbacks': handlers})
    trace_id = langfuse.get_current_trace_id()   # 이 span 이 속한 trace 의 id

langfuse.flush()   # 기록은 모아서 보낸다 - 지금 보내라고 한 번 밀어 준다

my_cost = cost_usd(checked.usage_metadata)
print('토큰    :', checked.usage_metadata['input_tokens'], '+', checked.usage_metadata['output_tokens'])
print('내 계산 : $', f'{my_cost:.8f}')
print('trace id:', trace_id)

이제 서버에 올라간 그 실행을 다시 꺼내 옵니다. **전송에는 몇 초 지연이 있어서** 바로 조회하면 아직 없다고 나옵니다. 그래서 **모델 호출 기록이 붙을 때까지** 잠깐 기다렸다 읽습니다. 비교할 값은 그 **모델 호출 하나의 비용**입니다.

In [ ]:
# 전송에는 지연이 있다 - 서버에 나타날 때까지 몇 초 기다렸다 읽는다
import time


def wait_for_generation(wanted_id, usage, tries=15):
    """방금 그 호출의 기록이 서버에 붙을 때까지 기다렸다가 돌려준다."""
    for attempt in range(tries):
        try:
            found = langfuse.api.trace.get(wanted_id)
            # trace 껍데기가 먼저 오고 모델 호출 기록은 조금 늦게 붙는다
            for observation in found.observations:
                # 토큰 수로 우리가 방금 부른 그 호출을 집어낸다
                if (observation.type == 'GENERATION'
                        and observation.usage.input == usage['input_tokens']
                        and observation.usage.output == usage['output_tokens']
                        and observation.calculated_total_cost):
                    return observation
        except Exception:
            pass
        time.sleep(3)
    raise RuntimeError('기록이 서버에 보이지 않습니다. 대시보드에서 직접 확인해 보세요.')


generation = wait_for_generation(trace_id, checked.usage_metadata)
server_cost = generation.calculated_total_cost

print('모델        :', generation.model)          # 단가는 이 이름으로 정해진다
print('토큰(서버)  :', generation.usage.input, '+', generation.usage.output)
print('토큰(응답)  :', checked.usage_metadata['input_tokens'], '+', checked.usage_metadata['output_tokens'])
print('비용(서버)  : $', f'{server_cost:.8f}')
print('비용(내 계산): $', f'{my_cost:.8f}')
print('두 값의 차이 : $', f'{abs(server_cost - my_cost):.10f}')

> 두 값이 소수점 아래까지 같습니다. Langfuse 도 **같은 요금표에 토큰 수를 곱했을 뿐**이기 때문입니다. 그래서 비용을 줄이는 일은 결국 **토큰을 줄이는 일**이 됩니다. 어느 프롬프트가 토큰을 많이 먹는지는 이제 대시보드에서 실행별로 볼 수 있습니다.

### 🖐️ 함께 따라하기: 긴 답과 짧은 답의 비용 차이

같은 주제를 **길게** 물은 답과 **한 문장으로** 물은 답의 비용을 각각 계산해 비교해 보세요.

1. `'벡터 검색을 자세히 설명해줘'` 를 부르고 `cost_usd` 로 비용을 구합니다.
2. `'벡터 검색을 한 문장으로 설명해줘'` 도 같은 방법으로 구합니다.
3. 두 비용을 함께 출력합니다.

**확인 기준**: 둘 다 $0.0000x 수준의 아주 작은 값이고, **긴 쪽이 몇 배 비쌉니다.** 출력 단가가 입력의 네 배라, 길게 답하게 할수록 요금이 빨리 오릅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) '벡터 검색을 자세히 설명해줘' 를 invoke 해 cost_usd 로 비용을 구한다
# 2) '벡터 검색을 한 문장으로 설명해줘' 도 같은 방법으로 구한다
# 3) 두 비용을 함께 출력한다

### ✅ 바로 확인 퀴즈

**1.** 입력 1,000 토큰·출력 500 토큰을 쓴 호출의 비용은 얼마인가요? (입력 $0.15 / 출력 $0.60 per 1M)

<details><summary>정답 보기</summary>

1000 ÷ 1,000,000 × 0.15 = **$0.00015**, 500 ÷ 1,000,000 × 0.60 = **$0.0003**. 합쳐서 **$0.00045** 입니다. **출력이 절반인데 비용은 두 배**라는 점을 보세요.

</details>

**2.** 내가 계산한 비용과 Langfuse 가 보여 주는 비용이 다르면 무엇을 의심해야 하나요?

<details><summary>정답 보기</summary>

**단가표**입니다. 코드에 적어 둔 값이 옛 요금이거나, 실제로 불린 모델이 내가 생각한 모델이 아닐 수 있습니다. trace 의 generation 에 찍힌 **모델 이름**을 먼저 확인하세요.

</details>

---
# 4. 프롬프트를 코드 밖에서 관리하기

## 왜 코드 밖으로 뺄까요?
프롬프트를 코드에 박아 두면, 문구 하나 바꿔도 **코드를 다시 배포**해야 합니다. 프롬프트를 **파일(또는 서버)** 에 **버전별로** 두면, 코드는 그대로 두고 **프롬프트만 v1 → v2 로 교체**할 수 있습니다. 어떤 버전이 더 좋은지 **비교 실험**도 쉬워집니다.

## 4.1 감 잡기: 파일에 두기
먼저 가장 단순한 형태로 감을 잡습니다. 프롬프트를 일차 폴더의 `data/prompts.yaml` 에 두고 이름·버전으로 꺼냅니다. 여기까지가 "코드 밖"의 최소 형태입니다. 다만 파일은 **내 노트북 안**에만 있어서, 팀이 함께 쓰거나 **어느 버전이 실제로 서비스에 나갔는지** 남기기 어렵습니다. 그래서 실무에서는 이 역할을 **서버**가 맡습니다. 그게 4.2 부터 쓸 **Langfuse 프롬프트 관리**입니다.

In [ ]:
# [제공 코드] 프롬프트 버전 레지스트리(로컬) - 실행만 하세요.
# 프롬프트를 코드가 아니라 파일(data/prompts.yaml)에서 불러오면, 코드 배포 없이 프롬프트만 교체할 수 있습니다.
# Langfuse 를 쓰면 langfuse.get_prompt(이름, version=번호) 가 똑같은 일을 서버에서 해 줍니다.
from pathlib import Path

import yaml

# 노트북 위치에 따라 data 폴더가 몇 단계 위인지 달라집니다(일차 폴더 / 교안 폴더 / 교안 폴더 안의 정답).
_PROMPTS_PATH = Path("data/prompts.yaml")
if not _PROMPTS_PATH.exists():
    _PROMPTS_PATH = Path("../data/prompts.yaml")
if not _PROMPTS_PATH.exists():
    _PROMPTS_PATH = Path("../../data/prompts.yaml")
PROMPTS = yaml.safe_load(_PROMPTS_PATH.read_text(encoding="utf-8"))


def get_prompt(name, version):
    """이름·버전으로 프롬프트 문자열을 돌려준다(로컬 버전 사전에서)."""
    return PROMPTS[name][version]


In [ ]:
# 파일에서 두 버전을 꺼내 본다 - '이름 + 버전으로 프롬프트를 부른다'는 감만 잡으면 된다
print('[v1]', get_prompt('insight_writer', 'v1'))
print()
print('[v2]', get_prompt('insight_writer', 'v2'))

## 4.2 서버에 올리기: `create_prompt`

이제 같은 일을 **Langfuse 서버**에서 합니다. 프롬프트의 집이 파일이 아니라 서버가 되면, 팀이 대시보드에서 함께 고치고, **어떤 버전이 언제 나갔는지**가 기록으로 남습니다.

프롬프트 본문에 `{{summary}}` 처럼 **이중 중괄호**를 쓰면 그 자리가 **채울 변수**가 됩니다. 값을 채우는 것은 뒤에서 볼 `.compile(summary=...)` 입니다.

> **주의**: 4.1 의 로컬 `get_prompt` 와 서버 메서드는 **이름이 같습니다**. 헷갈리지 않도록 서버 쪽은 항상 `langfuse.get_prompt(...)` 처럼 **클라이언트를 앞에 붙여** 부릅니다.

In [ ]:
# 프롬프트의 집을 서버로 옮긴다 - langfuse 클라이언트는 2절에서 인증을 확인한 그 클라이언트다
langfuse = get_client()

PROMPT_NAME = 'insight-report-writer'

first_prompt = langfuse.create_prompt(
    name=PROMPT_NAME,
    prompt='너는 데이터 분석 리포트 작성자다. 아래 수치를 바탕으로 인사이트를 한 문단으로 써라.\n\n수치: {{summary}}',
    labels=['production'])   # production 라벨 = 서비스가 기본으로 가져다 쓰는 버전
print('올린 버전:', first_prompt.version, '| 라벨:', first_prompt.labels)

이제 문구를 고친 **다음 버전**을 올립니다. 같은 이름으로 다시 올리면 **덮어쓰기가 아니라 새 버전이 쌓입니다**. 옛 버전은 그대로 남아 언제든 되돌아갈 수 있습니다(그래서 이 셀을 다시 실행하면 버전 번호가 3, 4 로 계속 올라갑니다).

In [ ]:
# 같은 이름으로 한 번 더 올린다 - 덮어쓰기가 아니라 '새 버전'이다
second_prompt = langfuse.create_prompt(
    name=PROMPT_NAME,
    prompt='너는 데이터 분석 리포트 작성자다. 아래 수치를 바탕으로 인사이트를 쓰되 '
           '1) 핵심 수치를 다시 말하고 2) 그 의미를 해석하고 3) 실행 가능한 제안을 한 문장 덧붙여라. '
           '관찰된 데이터일 뿐 인과관계 주장은 피하라.\n\n수치: {{summary}}',
    labels=['production'])   # 라벨을 다시 붙이면 production 이 이 새 버전으로 옮겨 온다
print('올린 버전:', second_prompt.version, '| 라벨:', second_prompt.labels)
print('앞 버전과 본문이 다른가:', first_prompt.prompt != second_prompt.prompt)

**확인 기준**: [cloud.langfuse.com](https://cloud.langfuse.com) 의 **Prompts** 탭을 열면 `insight-report-writer` 가 보이고, 그 안에 방금 올린 **두 개의 버전**이 쌓여 있습니다. 최신 버전에 **production** 라벨이 붙어 있는지 함께 보세요.

## 4.3 불러오기: `get_prompt`

부르는 방법은 세 가지입니다.

| 부르는 법 | 무엇을 주나 |
|---|---|
| `langfuse.get_prompt(이름)` | **production 라벨이 붙은 버전**(번호가 가장 큰 버전이 아닙니다) |
| `langfuse.get_prompt(이름, version=번호)` | 그 **번호의 버전**(옛 버전 고정) |
| `langfuse.get_prompt(이름, label='production')` | 라벨을 **명시**해서 (첫 줄과 같은 뜻) |

라벨을 쓰면 코드는 `production` 만 부르고, **어느 버전을 내보낼지는 대시보드에서 라벨만 옮겨** 정할 수 있습니다.

> `cache_ttl_seconds=0` 은 **캐시를 끄는** 옵션입니다. 기본은 한 번 받은 프롬프트를 60초 재사용하는데, 수업처럼 **방금 올린 버전을 곧바로** 부르면 옛 버전이 나올 수 있어 여기서는 꺼 둡니다.

In [ ]:
# 세 가지 방법으로 불러와 무엇이 오는지 비교한다
production = langfuse.get_prompt(PROMPT_NAME, cache_ttl_seconds=0)
first = langfuse.get_prompt(PROMPT_NAME, version=first_prompt.version, cache_ttl_seconds=0)
labeled = langfuse.get_prompt(PROMPT_NAME, label='production', cache_ttl_seconds=0)

print('라벨·버전 없이 부른 것 :', production.version)
print('버전을 지정해 부른 것  :', first.version)
print("label='production' 지정:", labeled.version)
print('두 버전의 본문이 다른가:', production.prompt != first.prompt)
print('채울 변수:', production.variables)

## 4.4 `.compile()` 로 값을 채워 실제로 호출하기

불러온 프롬프트는 문자열이 아니라 **객체**입니다. `.compile(변수=값)` 을 부르면 `{{변수}}` 자리가 채워진 **완성된 프롬프트 문자열**이 나오고, 그걸 그대로 모델에 넣습니다. 아래에서 **옛 버전과 production 버전의 응답을 비교**합니다. 코드는 한 줄도 안 바꾸고 **서버의 버전만** 바꿔 실험하는 것입니다.

In [ ]:
# 같은 입력에 두 버전을 각각 적용해 응답 차이를 본다(호출에는 관측 콜백도 붙인다)
summary = '카테고리별 평균 완료율: 파이썬 70.5%, 데이터분석 61.7%, AI머신러닝 54.8%.'

for prompt in [first, production]:
    filled = prompt.compile(summary=summary)   # {{summary}} 자리에 값을 채운다
    r = model.invoke(filled, config={'callbacks': handlers})
    print(f'=== 버전 {prompt.version} ===')
    print(r.text)
    print()

> production 버전은 "수치 재진술·해석·실행 제안"을 지시하므로 더 구조화된 답이 나올 것입니다. 위 4.3 출력의 **두 버전의 본문이 다른가** 가 참인지 먼저 확인하고 두 응답을 견줘 보세요. **바뀐 것은 서버의 프롬프트뿐이고 코드는 그대로**라는 점이 핵심입니다.

## 4.5 트레이스에 "어느 버전을 썼는지" 남기기

관측과 프롬프트 관리가 만나는 자리입니다. 나중에 "이 이상한 답은 **어느 프롬프트 버전**이 만든 거지?" 를 답하려면, **trace 에 버전이 붙어 있어야** 합니다. 프롬프트를 LangChain 템플릿으로 바꿔 끼우면서 `metadata={'langfuse_prompt': 프롬프트}` 를 주면 연결됩니다.

`get_langchain_prompt()` 는 Langfuse 표기(`{{summary}}`)를 LangChain 표기(`{summary}`)로 바꿔 줍니다.

In [ ]:
# 프롬프트를 템플릿으로 끼우고, 그 템플릿에 '이 프롬프트에서 왔다'는 표시를 붙인다
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template(
    production.get_langchain_prompt(),          # {{summary}} -> {summary} 표기로 바꿔 준다
    metadata={'langfuse_prompt': production})   # 이 한 줄이 trace 와 프롬프트 버전을 잇는다
chain = template | model
r = chain.invoke({'summary': summary}, config={'callbacks': handlers})
langfuse.flush()   # 대시보드에서 바로 보이도록 전송을 밀어낸다
print(r.text)

**확인 기준**: Langfuse 의 **Tracing → Traces** 에서 방금 실행을 열면 모델 호출(generation)에 `insight-report-writer` 의 **버전**이 함께 표시됩니다. 반대로 **Prompts** 탭에서 그 프롬프트를 열면 그것을 쓴 실행들을 되짚어 볼 수 있습니다. (표시가 바로 안 보이면 잠시 뒤 새로고침하세요.)

### 🖐️ 함께 따라하기: 다른 프롬프트를 올리고 라벨로 불러오기

이번에는 리포트가 아니라 **강의 소개 문구**를 쓰는 프롬프트를 새로 올려 봅니다. 이름은 `'course-summary-writer'`, 본문은 아래와 같이, 라벨은 `['production']` 으로 올린 뒤 **이름과 라벨만으로** 다시 불러와 `.version` 과 `.variables` 를 출력하세요.

```text
너는 강의 소개 문구 작성자다. 아래 강의를 한 문장으로 소개하라.

강의: {{course}}
```

**확인 기준**: 이 이름을 처음 올렸다면 버전은 1 이고(셀을 다시 실행하면 2, 3 으로 올라갑니다), 변수 목록은 `['course']` 입니다. 앞 데모의 `summary` 가 아니라 **이 프롬프트가 쓰는 변수**가 나온다는 점을 확인하세요. 불러올 때 버전 숫자를 적지 않았다는 점도요. **라벨만 옮기면 코드를 배포하지 않고도 프롬프트를 바꿀 수 있다**는 것이 프롬프트 버전관리의 핵심입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) langfuse.create_prompt 로 'course-summary-writer' 를 올린다(labels=['production'])
#    본문은 위 마크다운의 두 줄을 그대로 쓴다(줄바꿈은 \n\n, 변수 자리는 {{course}})
# 2) langfuse.get_prompt(이름, label='production', cache_ttl_seconds=0) 으로 되불러온다
# 3) 되불러온 프롬프트의 .version 과 .variables 를 출력한다

### ✅ 바로 확인 퀴즈

**1.** 프롬프트를 코드 밖 파일/서버에서 버전 관리하면 좋은 점은?

<details><summary>정답 보기</summary>

**코드 재배포 없이 프롬프트만 교체·비교**할 수 있습니다. Langfuse 에서는 `create_prompt` 로 올리고 `get_prompt` 로 불러 씁니다.

</details>

**2.** 같은 이름으로 `create_prompt` 를 한 번 더 부르면 어떻게 되나요?

<details><summary>정답 보기</summary>

덮어쓰지 않고 **새 버전이 쌓입니다**(옛 버전은 그대로 남습니다). `labels=['production']` 을 함께 주면 **production 라벨이 새 버전으로 옮겨 갑니다**.

</details>

**3.** `langfuse.get_prompt('이름')` 을 버전·라벨 없이 부르면 어떤 버전이 오나요?

<details><summary>정답 보기</summary>

**production 라벨이 붙은 버전**입니다(번호가 가장 큰 버전이 아닙니다). 그래서 대시보드에서 라벨만 옮기면 코드를 건드리지 않고 내보낼 버전을 바꿀 수 있습니다.

</details>

---
# 5. 여러 호출을 한 세션으로 묶기

지금까지는 호출 하나가 trace 하나였습니다. 그런데 챗봇은 **한 사람이 이어서 여러 번** 묻습니다. 그 대화를 나중에 통째로 열어 보려면 trace 들을 한 묶음으로 표시해 두어야 합니다. 그 묶음이 **세션(session)** 입니다.

| 단위 | 묶는 범위 |
|---|---|
| generation | 모델 호출 하나 |
| trace | 요청 하나(그 안의 단계·도구 호출까지) |
| **session** | **같은 사용자의 여러 요청, 곧 대화 한 판** |

붙이는 방법은 콜백과 같은 자리입니다. `config` 의 **`metadata` 에 `langfuse_session_id` 를 실어** 보내면 **같은 값을 준 호출끼리 한 세션**이 됩니다. 사람마다 다른 값을 주면 사용자별로도 갈라 볼 수 있습니다.

In [ ]:
# 대화 한 판에 이름표를 붙인다 - 같은 이름표를 단 호출이 대시보드에서 한 세션으로 묶인다
session_id = 'day22-demo-session'

questions = ['관측성을 한 문장으로 설명해줘',
             '방금 설명에서 trace 는 무엇을 가리켜?',
             '그럼 session 은 언제 쓰지?']
for question in questions:
    answer = model.invoke(question,
                          config={'callbacks': handlers,
                                  'metadata': {'langfuse_session_id': session_id}})
    print('-', answer.text[:50], '...')

langfuse.flush()
print()
print('세션 이름표:', session_id)

**확인 기준**: [cloud.langfuse.com](https://cloud.langfuse.com) 의 **Tracing → Sessions** 에 `day22-demo-session` 이 한 줄로 보이고, 열면 방금 세 번의 호출이 **순서대로** 들어 있습니다.

코드로도 확인할 수 있습니다. 세션 이름표로 조회하면 그 안에 묶인 trace 가 나옵니다.

In [ ]:
# 세션에 무엇이 묶였는지 서버에서 되읽어 본다(여기서도 전송 지연을 기다린다)
session = None
for attempt in range(10):
    try:
        session = langfuse.api.sessions.get(session_id)
        if len(session.traces) >= len(questions):
            break
    except Exception:
        pass
    time.sleep(3)

print('세션에 묶인 실행 수:', len(session.traces) if session else 0)
for trace_in_session in (session.traces if session else []):
    print(' -', trace_in_session.name, str(trace_in_session.timestamp)[:19])

> 세션이 없으면 대화 한 판이 **흩어진 실행 여러 개**로 남습니다. 사용자가 "어제 그 대화에서 이상한 답을 받았다"고 할 때, 세션이 있으면 그 대화 하나만 열면 되고 없으면 시간대를 뒤져야 합니다.

### 🖐️ 함께 따라하기: 두 사람의 대화를 갈라 놓기

세션 이름표를 `'demo-user-a'` 로 주고 두 번, `'demo-user-b'` 로 주고 한 번 호출해 보세요. 질문은 아무거나 좋습니다.

**확인 기준**: 대시보드 **Sessions** 에 두 줄이 생기고, 각각 **2건과 1건**이 묶여 있습니다. 코드는 그대로이고 **이름표만 달리 준 것**이 요점입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 'demo-user-a' 를 langfuse_session_id 로 주고 두 번 invoke 한다
# 2) 'demo-user-b' 를 주고 한 번 invoke 한다
# 3) flush() 로 밀어 보낸 뒤 대시보드 Sessions 에서 두 줄을 확인한다

### ✅ 바로 확인 퀴즈

**1.** trace 와 session 은 무엇이 다른가요?

<details><summary>정답 보기</summary>

**trace 는 요청 하나**이고, **session 은 그 요청들을 묶은 대화 한 판**입니다. 챗봇에서 사용자가 세 번 물으면 trace 는 셋, session 은 하나입니다.

</details>

**2.** 세션으로 묶으려면 호출 코드에서 무엇을 바꾸나요?

<details><summary>정답 보기</summary>

로직은 그대로 두고 `config` 에 **`metadata={'langfuse_session_id': 값}`** 을 함께 넘깁니다. 콜백과 같은 자리입니다. 같은 값을 준 호출끼리 한 세션이 됩니다.

</details>

---
# 6. 실패를 추적하는 워크플로

기록이 남아 있으면 **실패를 되짚기**가 쉬워집니다. 흔한 실패 하나를 봅시다. **도구 설명이 부실하면** 에이전트가 **엉뚱한 도구를 고르거나 도구를 안 씁니다**(에이전트 단원의 라우팅 문제). 이럴 때 **메시지 기록**을 읽어 **어디서 틀어졌는지** 찾습니다. 아래 실행에는 2절의 `handlers` 를 그대로 붙여, 같은 메시지 기록이 **Langfuse 대시보드에도** 남게 합니다. 수업에서는 노트북 출력으로 보지만, 운영에서는 학생이 옆에 없으니 대시보드에 남은 trace 가 유일한 단서가 됩니다.

**확인 기준**: 아래 두 실행을 마친 뒤 Langfuse 의 **Tracing → Traces** 에서 각각을 열어, **어느 도구가 trace 안의 단계로 찍혔는지**가 노트북 출력과 같은지 확인하세요.

## 디버깅 순서
1. **메시지 기록을 연다**: 어떤 도구가 불렸는지(`tool_calls`)·안 불렸는지 본다.
2. **원인을 좁힌다**: 도구 설명(docstring)이 모호하지 않은지, 프롬프트가 잘못되지 않았는지 본다.
3. **고치고 재현한다**: 설명을 명확히 하고 같은 질문으로 다시 돌려 라우팅이 바뀌는지 확인한다.

아래에서 이 순서를 **직접 재현**합니다. 설명이 모호한 도구로 라우팅을 **일부러 틀어 보고**, 설명만 고쳐 같은 질문을 다시 돌립니다.

In [ ]:
# 메시지 기록에서 '어떤 도구가 불렸나'를 뽑는 도우미 - 실패 추적의 첫 단계
from langchain_core.messages import AIMessage


def tool_trace(result):
    names = []
    for message in result['messages']:
        # 도구 호출은 AIMessage 에만 담긴다(사람 말·도구 결과에는 없다) - 타입으로 가른다
        if isinstance(message, AIMessage) and message.tool_calls:
            for call in message.tool_calls:
                names.append(call['name'])
    return names

## 6.1 라우팅을 일부러 틀어 보기

수강료를 돌려주는 도구인데 이름도 설명도 **모호하게**(`lookup` · "정보를 알려준다") 두고, 옆에 그럴듯한 다른 도구(`course_guide`)를 함께 줍니다. 에이전트는 무엇을 고를까요?

In [ ]:
# 실패 재현 - 진짜 필요한 도구의 이름·설명이 모호하면 라우팅이 틀어진다
from langchain.agents import create_agent
from langchain_core.tools import tool

PRICES = {'파이썬': 79000, '데이터분석': 99000, 'AI머신러닝': 129000}


def price_of(category):
    """'데이터분석 강의' 처럼 이름이 섞여 들어와도 찾도록 부분 일치로 고른다."""
    for name, price in PRICES.items():
        if name in category:
            return price
    return 49000


@tool
def lookup(category: str) -> str:
    """정보를 알려준다."""        # ← 무엇을 언제 쓰는 도구인지 알 수 없다
    return str(price_of(category))

@tool
def course_guide(category: str) -> str:
    """강의에 대해 안내한다."""
    return f'{category} 강의는 초급자를 위한 과정입니다.'

question = '데이터분석 강의 수강료는?'
vague_agent = create_agent(model, [lookup, course_guide])
# config 한 자리를 얹으면 이 메시지 기록이 대시보드에도 그대로 올라간다 - 관측과 디버깅이 여기서 만난다
vague_run = vague_agent.invoke({'messages': [{'role': 'user', 'content': question}]},
                               config={'callbacks': handlers})
print('불린 도구:', tool_trace(vague_run))
print('최종 답 :', vague_run['messages'][-1].text)

> 대개 **엉뚱한 `course_guide` 를 먼저 부르거나**(그만큼 호출·토큰이 낭비됩니다) 수강료를 못 찾은 답이 나옵니다. 본인 화면의 목록을 그대로 읽어 두세요. 다음 셀과 **비교**할 것입니다. (모델은 매번 똑같이 굴지 않으므로 한 번에 제대로 부를 때도 있습니다.)

## 6.2 설명만 고쳐 다시 돌리기

로직·질문·모델은 **하나도 바꾸지 않고**, 도구의 **이름과 설명만** 명확히 합니다.

In [ ]:
# 고친 뒤 재현 - 바꾼 것은 도구의 이름과 docstring 한 줄뿐이다
@tool
def course_price(category: str) -> int:
    """강의 카테고리 이름을 받아 대표 수강료(원)를 돌려준다."""
    return price_of(category)

fixed_agent = create_agent(model, [course_price, course_guide])
fixed_run = fixed_agent.invoke({'messages': [{'role': 'user', 'content': question}]},
                               config={'callbacks': handlers})
print('불린 도구:', tool_trace(fixed_run))
print('최종 답 :', fixed_run['messages'][-1].text)

> 두 출력을 나란히 보세요. 고친 쪽은 **`course_price` 한 번**으로 끝나고 답에 수강료가 들어 있습니다. 도구가 안 불렸거나 엉뚱한 게 불렸을 때 **가장 먼저 의심할 곳이 도구 설명(docstring)** 인 이유입니다. 노트북을 닫은 뒤에도 이 두 메시지 기록을 대시보드에서 다시 꺼내 비교할 수 있다는 것이 관측 도구를 붙이는 이유입니다.

### ✅ 바로 확인 퀴즈

**1.** 에이전트가 엉뚱한 답을 냈을 때 **가장 먼저** 확인할 것은 무엇인가요?

<details><summary>정답 보기</summary>

**메시지 기록에서 어떤 도구가 불렸는지**(`tool_calls`)를 봅니다. 도구가 안 불렸거나 엉뚱한 것이 불렸다면 그다음 의심할 곳은 **도구의 이름과 설명(docstring)** 입니다.

</details>

**2.** 수업에서는 두 실행을 노트북 출력으로 비교했습니다. 운영 중인 서비스에서는 무엇으로 비교하나요?

<details><summary>정답 보기</summary>

관측 도구에 쌓인 **trace** 로 비교합니다. 노트북 출력은 커널을 닫으면 사라지지만, `handlers` 를 붙여 둔 실행은 Langfuse 에 남아 나중에 다시 열어 볼 수 있습니다. 관측을 붙여 두는 이유가 여기 있습니다.

</details>

---
## 이번 강의 정리

| 주제 | 하는 일 | 핵심 |
|---|---|---|
| 관측성 | 요청을 계층으로 기록 | trace → span → generation |
| 콜백 | 호출을 자동 전송 | `config={'callbacks': handlers}` |
| 토큰·비용 | 읽고, 계산하고, 서버 값과 대조 | `usage_metadata` · 100만 토큰당 단가 · `trace.total_cost` |
| 프롬프트 관리 | 서버에 버전으로 두기 | `langfuse.create_prompt` / `langfuse.get_prompt(label='production')` |
| 세션 | 대화 한 판으로 묶기 | `metadata={'langfuse_session_id': 값}` |
| 실패 추적 | 메시지 기록으로 라우팅 확인 | `tool_calls` 를 읽고 도구 설명(docstring)을 고친다 |

- 관측성 = 밖에 남긴 **기록만 보고 안에서 무슨 일이 있었는지 알아낼 수 있는 성질**. 비용·성능·실패 원인·버전 비교가 그 기록으로 답하는 질문들입니다. 대표 도구는 **Langfuse** 와 **LangSmith** 이고, 사내 서버에 직접 올릴 수 있고 프레임워크에 묶이지 않아 이 수업은 Langfuse 를 썼습니다.
- 비용은 **토큰 × 단가**입니다. 서버가 보여 주는 값도 같은 계산이라, 줄이려면 **토큰을 줄여야** 합니다.
- 관측을 붙이는 자리는 언제나 **`config` 한 곳**이었습니다. 콜백도, 세션 이름표도, 프롬프트 버전 연결도 거기였습니다.

## ⏭️ 예고

이제 에이전트를 **만들고(도구·루프)**, **자동화하고(분석·성찰)**, **들여다볼(관측·디버깅)** 수 있습니다. 다음 단원에서는 이 에이전트를 사람이 쓰는 **웹 화면(Streamlit 대시보드·챗봇)** 으로 감싸 봅니다.

수고하셨습니다!